# Router2 Summary Visualization

Use this notebook to inspect the Router2 and RouterDC results saved in this folder.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

## Load Results

In [ ]:
ROOT = Path.cwd().resolve()
if (ROOT / "router2_summary.csv").exists():
    SUMMARY_DIR = ROOT
elif (ROOT / "results" / "router2_summary" / "router2_summary.csv").exists():
    SUMMARY_DIR = ROOT / "results" / "router2_summary"
elif (ROOT.parent / "results" / "router2_summary" / "router2_summary.csv").exists():
    SUMMARY_DIR = ROOT.parent / "results" / "router2_summary"
else:
    raise FileNotFoundError("Could not find results/router2_summary/router2_summary.csv")

summary_path = SUMMARY_DIR / "router2_summary.csv"
results_df = pd.read_csv(summary_path)

for column in ("test_mae", "test_mse", "test_rmse"):
    results_df[column] = pd.to_numeric(results_df[column], errors="coerce")

print(f"Loaded {len(results_df)} rows from {summary_path}")

## All Results

In [ ]:
all_results_view = (
    results_df[
        [
            "method",
            "result_type",
            "test_mae",
            "test_mse",
            "test_rmse",
        ]
    ]
    .sort_values(["test_mae", "method"])
    .round(6)
)

display(all_results_view)

## Best Deployable Methods

In [ ]:
deployable_df = results_df[~results_df["result_type"].eq("oracle")].copy()
best_methods_df = (
    deployable_df.sort_values("test_mae")
    .drop_duplicates(subset=["method", "result_type"], keep="first")
    .head(10)
)

display(
    best_methods_df[
        ["method", "result_type", "test_mae", "test_mse", "test_rmse"]
    ].round(6)
)

## Trained Routers Only

In [ ]:
trained_router_df = (
    results_df[results_df["result_type"].eq("trained_router")]
    .copy()
    .sort_values("test_mae")
)

display(
    trained_router_df[
        ["method", "test_mae", "test_mse", "test_rmse"]
    ].round(6)
)

## Chart: Best Deployable Methods

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(
    best_methods_df["method"],
    best_methods_df["test_mae"],
    color="#2f6f73",
)

ax.set_title("Best Deployable Router2 Methods")
ax.set_xlabel("Method")
ax.set_ylabel("Test MAE, lower is better")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=20)

for bar, value in zip(bars, best_methods_df["test_mae"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
chart_path = SUMMARY_DIR / "router2_best_deployable_methods.png"
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path}")

## Chart: Trained Routers

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.barh(
    trained_router_df["method"],
    trained_router_df["test_mae"],
    color="#7a4f9f",
)

ax.set_title("Trained Router2 / RouterDC Test MAE")
ax.set_xlabel("Test MAE, lower is better")
ax.grid(axis="x", alpha=0.25)
ax.invert_yaxis()

for bar, value in zip(bars, trained_router_df["test_mae"]):
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value:.4f}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
chart_path = SUMMARY_DIR / "router2_trained_routers_mae.png"
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path}")

## Chart: Soft Router2 Vs Baselines

In [ ]:
soft_methods = [
    "Fixed validation-based soft weights",
    "Fixed equal average",
    "Router 2 feature router",
    "Multiscale TCN expert-embedding router",
]

soft_df = results_df[
    results_df["router_family"].eq("soft_router2")
    & results_df["method"].isin(soft_methods)
]
soft_df = soft_df.drop_duplicates(subset=["method"], keep="first")

soft_comparison_df = soft_df[["method", "result_type", "test_mae", "test_mse", "test_rmse"]].sort_values("test_mae")
display(soft_comparison_df.round(6))

ax = soft_comparison_df.plot(x="method", y="test_mae", kind="bar", figsize=(11, 5), width=0.8, legend=False)
ax.set_title("Soft Router2 vs Simple Baselines")
ax.set_xlabel("Method")
ax.set_ylabel("Test MAE, lower is better")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
chart_path = SUMMARY_DIR / "router2_soft_vs_baselines.png"
plt.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path}")

## Files In This Folder

In [ ]:
for path in sorted(SUMMARY_DIR.iterdir()):
    if path.is_file():
        print(path.name)